# 🍳 Actividad 10 — MCP Avanzado: Tools, Prompts, Sampling y Elicitation

## ¿Qué construirás?

Un **servidor MCP** (Model Context Protocol de Anthropic) para un asistente de recetas de cocina,
y un **cliente Gradio** que explora los cuatro mecanismos avanzados del protocolo:

| Parte | Concepto | ¿Qué hace? |
|-------|----------|----------|
| 3–4 | **Tools + Prompts** | El cliente llama funciones y obtiene plantillas del servidor |
| 5   | **Sampling** | El servidor le pide al cliente que use Ollama para generar texto |
| 6   | **Elicitation** | El servidor solicita datos al usuario a través del cliente |
| 7   | **Integración** | Todo junto en una interfaz Gradio con 4 pestañas |

## Arquitectura final

```
  Gradio UI (4 pestañas)
       │
       ▼
  MCP Client (Python)  ←──────── sampling_callback (llama a Ollama)
       │ HTTP / SSE              elicitation_callback (captura formulario Gradio)
       ▼
  MCP Server (mcp_server.py) — puerto 8000
  ├── buscar_receta          ── Tool básica (datos locales)
  ├── calcular_calorias      ── Tool básica (cálculo local)
  ├── sugerir_maridaje       ── Tool básica (tabla local)
  ├── analizar_receta_con_ia ── Tool + SAMPLING  (delega texto a Ollama)
  ├── adaptar_receta         ── Tool + ELICITATION (pide datos al usuario)
  ├── prompt_chef_experto    ── Prompt (plantilla parametrizada)
  └── prompt_adaptacion_dieta── Prompt (plantilla parametrizada)
```


## ⚙️ PARTE 1 — Instalación y configuración del entorno

In [ ]:
# ============================================================
# CELDA 1.1 — Verificar GPU disponible
# ============================================================
# Ollama necesita GPU para correr el modelo en tiempo razonable.
# Sin GPU T4 activa, los modelos tardan 10-20x más.

import subprocess

resultado_gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if resultado_gpu.returncode != 0:
    raise RuntimeError(
        '❌ No se detectó GPU.\n'
        'Ve a Entorno de ejecución → Cambiar tipo → GPU T4 y vuelve a ejecutar.'
    )

print('✅ GPU disponible:')
print(resultado_gpu.stdout.split('\n')[8])  # Línea con el modelo de GPU


In [ ]:
# ============================================================
# CELDA 1.2 — Instalar dependencias del sistema y Ollama
# ============================================================
# zstd: para descomprimir el binario de Ollama
# pciutils: para que Ollama detecte la GPU (sin esto corre en CPU)

import os
os.environ['DEBIAN_FRONTEND'] = 'noninteractive'  # Evita prompts interactivos de apt

print('📦 Instalando dependencias del sistema...')
!sudo apt-get update -qq 2>/dev/null
!sudo apt-get install -y -qq zstd pciutils 2>/dev/null

print('📥 Instalando Ollama...')
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -5

!ollama --version
print('✅ Ollama instalado')


In [ ]:
# ============================================================
# CELDA 1.3 — Instalar paquetes Python
# ============================================================
# mcp: el paquete oficial de Anthropic que implementa el protocolo MCP
# gradio, pyngrok, uvicorn: mismos que en actividades anteriores
# nest-asyncio: parchea el event loop de Colab para usar asyncio.run()

!pip install -q 'mcp>=1.9.0' gradio pyngrok nest-asyncio uvicorn

print('✅ Paquetes Python instalados')


In [ ]:
# ============================================================
# CELDA 1.4 — Iniciar el servidor Ollama en background
# ============================================================
# OLLAMA_HOST=0.0.0.0 permite conexiones desde localhost y procesos hijos.

import subprocess, time, requests

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'  # Acepta conexiones de cualquier interfaz

print('🚀 Iniciando servidor Ollama...')
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(5)  # Esperamos 5 segundos para que el servidor arranque

for intento in range(3):
    try:
        r = requests.get('http://localhost:11434', timeout=10)
        print(f'✅ Servidor Ollama activo: {r.text.strip()}')
        break
    except requests.ConnectionError:
        print(f'   Intento {intento + 1}/3... esperando')
        time.sleep(5)
else:
    raise RuntimeError('❌ El servidor Ollama no arrancó. Reinicia y vuelve a ejecutar.')


In [ ]:
# ============================================================
# CELDA 1.5 — Descargar el modelo llama3.2:1b
# ============================================================
# llama3.2:1b: modelo de 1.3 GB, rápido en T4, adecuado para demos.

MODELO_OLLAMA = 'llama3.2:1b'  # 🔧 PARÁMETRO: cambia por llama3.2:3b para más calidad

print(f'📥 Descargando {MODELO_OLLAMA} (~1.3 GB, puede tardar 2-4 minutos)...')
!ollama pull {MODELO_OLLAMA}
print(f'\n✅ Modelo {MODELO_OLLAMA} listo para usar')


In [ ]:
# ============================================================
# CELDA 1.6 — Importar y configurar el SDK de MCP
# ============================================================
# FastMCP: API de alto nivel de Anthropic para construir servidores MCP
# ClientSession: gestiona el protocolo cliente MCP
# sse_client: conecta al servidor por HTTP con transporte SSE
# mcp.types: tipos del protocolo: SamplingMessage, ElicitResult, etc.

from mcp.server.fastmcp import FastMCP, Context  # Servidor + contexto de ejecución
from mcp import ClientSession                     # Cliente MCP
from mcp.client.sse import sse_client             # Transporte SSE para el cliente
import mcp.types as types                         # Tipos del protocolo MCP
import mcp                                        # Para verificar la versión

import nest_asyncio   # Permite asyncio.run() dentro del event loop de Colab/IPython
nest_asyncio.apply()  # Parchea el event loop; se llama una sola vez al inicio

import asyncio        # Programación asíncrona (async/await)
import requests       # HTTP para llamar a Ollama desde el sampling_callback

print(f'✅ MCP versión: {mcp.__version__}')
print('✅ FastMCP, ClientSession, sse_client, types importados')
print('✅ nest_asyncio aplicado — asyncio.run() funcionará en Colab')


## 📚 PARTE 2 — Los cuatro conceptos avanzados de MCP

El paquete `mcp` de Anthropic implementa el **Model Context Protocol**, un estándar abierto
para comunicar agentes de IA con herramientas externas. Hasta la Actividad 09 solo usamos
**Tools**. Esta actividad cubre los cuatro mecanismos completos:

---

### 🔧 Tools — El cliente invoca, el servidor ejecuta

```
  Cliente  →  call_tool('buscar_receta', {...})  →  Servidor ejecuta
  Cliente  ←  resultado (texto)                 ←  Servidor
```

---

### 📋 Prompts — El cliente pide plantillas al servidor

```
  Cliente  →  get_prompt('prompt_chef_experto', {'tipo_cocina': 'italiana'})
  Cliente  ←  mensajes renderizados (system prompt listo para usar)
```

---

### 🧠 Sampling — El servidor le pide texto al cliente

```
  Cliente  →  call_tool('analizar_receta_con_ia', {...})
  Servidor necesita texto generado por IA...
  Servidor →  SamplingRequest (mensajes + max_tokens)  →  Cliente
  Cliente  →  sampling_callback → llama a Ollama
  Servidor ←  texto generado por Ollama               ←  Cliente
  Cliente  ←  resultado final de la tool              ←  Servidor
```
El servidor **no tiene LLM propio**. Delega al cliente, que decide qué modelo usar.

---

### 📝 Elicitation — El servidor pide datos al usuario

```
  Cliente  →  call_tool('adaptar_receta', {'nombre': 'paella'})
  Servidor necesita datos del usuario...
  Servidor →  ElicitRequest (mensaje + schema)  →  Cliente
  Cliente  →  elicitation_callback → formulario Gradio
  Usuario rellena: num_personas=6, alergias='gluten', nivel_picante='suave'
  Servidor ←  datos del formulario              ←  Cliente
  Cliente  ←  receta personalizada              ←  Servidor
```
El servidor define **qué datos necesita** (schema). El cliente define **cómo obtenerlos**.


## 🛠️ PARTE 3 — Servidor MCP v1: Tools básicas y Prompts

Escribimos el servidor con las 3 tools básicas y los 2 prompts.
En las Partes 5 y 6 añadiremos Sampling y Elicitation.

### ¿Por qué usar `%%writefile`?

El servidor corre como **proceso independiente** en el puerto 8000.
`%%writefile` escribe la celda directamente a un archivo `.py`,
que lanzamos con `subprocess.Popen`. Esta separación es la esencia de MCP:
el servidor no sabe nada del cliente.


In [ ]:
%%writefile mcp_server.py
# ============================================================
# mcp_server.py — Servidor MCP de Recetas v1 (tools + prompts)
# ============================================================
from mcp.server.fastmcp import FastMCP, Context
import mcp.types as types
from pydantic import BaseModel

mcp_server = FastMCP('chef-recetas')  # Nombre visible en el handshake del cliente


# ── BASE DE DATOS LOCAL DE RECETAS ────────────────────────────────────────
# Usamos datos locales para que la actividad funcione sin internet.

RECETAS = {
    'paella': {
        'nombre': 'Paella Valenciana', 'origen': 'Valencia, España',
        'ingredientes': ['arroz bomba 400g', 'pollo 500g', 'conejo 300g',
                         'judías verdes 150g', 'tomate maduro 2 uds',
                         'azafrán 1 sobre', 'aceite de oliva 100ml', 'sal al gusto'],
        'pasos': ['Calentar aceite en paellera a fuego medio',
                  'Dorar pollo y conejo troceados 10 minutos',
                  'Añadir judías verdes, sofreír 5 minutos',
                  'Incorporar tomate rallado, cocinar 5 minutos',
                  'Agregar arroz, azafrán y caldo (doble que arroz)',
                  'Cocer 18 minutos sin remover; reposar 5 minutos tapado'],
        'tiempo_min': 45, 'calorias_por_100g': 150,
        'maridaje': 'Vino blanco valenciano DO Valencia o agua con gas',
        'dificultad': 'media'
    },
    'tortilla': {
        'nombre': 'Tortilla Española', 'origen': 'España',
        'ingredientes': ['huevos 6 uds', 'patatas 500g', 'cebolla 1 ud',
                         'aceite de oliva 200ml', 'sal al gusto'],
        'pasos': ['Cortar patatas en láminas finas (3-4mm)',
                  'Freír patatas y cebolla en aceite bajo 20 minutos',
                  'Batir huevos con sal y mezclar con patatas escurridas',
                  'Cuajar en sartén antiadherente 4 minutos por cada lado'],
        'tiempo_min': 35, 'calorias_por_100g': 170,
        'maridaje': 'Vino tinto Rioja joven o cerveza española',
        'dificultad': 'fácil'
    },
    'gazpacho': {
        'nombre': 'Gazpacho Andaluz', 'origen': 'Andalucía, España',
        'ingredientes': ['tomates maduros 1kg', 'pepino 1 ud', 'pimiento verde 1 ud',
                         'ajo 2 dientes', 'pan del día anterior 100g',
                         'aceite de oliva virgen extra 100ml', 'vinagre de jerez 30ml', 'sal'],
        'pasos': ['Remojar el pan en agua fría 10 minutos',
                  'Triturar todos los ingredientes con batidora hasta crema fina',
                  'Colar por colador fino para eliminar pieles y pepitas',
                  'Refrigerar mínimo 2 horas antes de servir'],
        'tiempo_min': 20, 'calorias_por_100g': 35,
        'maridaje': 'Fino de Jerez muy frío o agua mineral',
        'dificultad': 'fácil'
    },
    'carbonara': {
        'nombre': 'Pasta Carbonara', 'origen': 'Roma, Italia',
        'ingredientes': ['spaghetti 400g', 'guanciale o panceta 200g',
                         'yemas de huevo 4 uds', 'queso pecorino romano 100g',
                         'pimienta negra molida', 'sal'],
        'pasos': ['Cocer pasta al dente, reservar 1 taza del agua de cocción',
                  'Dorar el guanciale en sartén sin aceite hasta crujiente',
                  'Mezclar yemas, queso rallado y pimienta en un bol',
                  'Unir pasta caliente con guanciale fuera del fuego',
                  'Añadir mezcla de yemas con agua de pasta para cremosidad'],
        'tiempo_min': 25, 'calorias_por_100g': 200,
        'maridaje': 'Vino blanco italiano Frascati o Pinot Grigio',
        'dificultad': 'media'
    },
    'hummus': {
        'nombre': 'Hummus Clásico', 'origen': 'Oriente Medio',
        'ingredientes': ['garbanzos cocidos 400g', 'tahini 60ml',
                         'limón (zumo) 3 cucharadas', 'ajo 2 dientes',
                         'aceite de oliva 3 cucharadas', 'comino 1 cucharadita', 'sal'],
        'pasos': ['Reservar el líquido de los garbanzos (aquafaba)',
                  'Triturar el ajo con sal hasta formar pasta',
                  'Añadir garbanzos, tahini, limón y comino; triturar',
                  'Agregar aquafaba poco a poco hasta textura cremosa'],
        'tiempo_min': 15, 'calorias_por_100g': 165,
        'maridaje': 'Té de menta caliente o agua mineral con gas',
        'dificultad': 'fácil'
    },
}


# ── TOOLS BÁSICAS ────────────────────────────────────────────────────────
# @mcp_server.tool() registra la función como herramienta MCP y genera
# el JSON Schema de parámetros a partir de los type hints de Python.
# El docstring se convierte en la descripción visible en list_tools().

@mcp_server.tool()
def buscar_receta(nombre: str, num_personas: int = 4) -> str:
    'Busca una receta de cocina y devuelve ingredientes y pasos de preparación.'
    receta = RECETAS.get(nombre.lower())  # Normalizamos a minúsculas
    if not receta:
        return f"No encontré '{nombre}'. Disponibles: {', '.join(RECETAS.keys())}"
    resultado = f"# {receta['nombre']}\n"
    resultado += f"**Origen:** {receta['origen']} | **Tiempo:** {receta['tiempo_min']} min | **Dificultad:** {receta['dificultad']}\n\n"
    resultado += f"## Ingredientes para {num_personas} personas\n"
    for ing in receta['ingredientes']:
        resultado += f'- {ing}\n'
    resultado += '\n## Preparación\n'
    for i, paso in enumerate(receta['pasos'], 1):
        resultado += f'{i}. {paso}\n'
    return resultado


@mcp_server.tool()
def calcular_calorias(plato: str, gramaje: int = 300) -> str:
    'Calcula las calorías aproximadas de una porción según el gramaje indicado.'
    receta = RECETAS.get(plato.lower())
    if not receta:
        return f"Sin datos para '{plato}'. Disponibles: {list(RECETAS.keys())}"
    calorias = int(receta['calorias_por_100g'] * gramaje / 100)  # Regla de tres
    clasificacion = 'Plato ligero ✅' if calorias < 400 else 'Plato contundente 🍽️'
    return (f"**{receta['nombre']}** — porción de {gramaje}g:\n"
            f'Calorías: ~{calorias} kcal\n'
            f'Base: {receta["calorias_por_100g"]} kcal/100g | {clasificacion}')


@mcp_server.tool()
def sugerir_maridaje(plato: str) -> str:
    'Sugiere la bebida ideal para acompañar el plato indicado.'
    receta = RECETAS.get(plato.lower())
    if not receta:
        return f"Sin maridaje para '{plato}'."
    return (f"**Maridaje para {receta['nombre']}:**\n"
            f"{receta['maridaje']}\n\n"
            f'💡 Un vino ácido limpia el paladar; uno con cuerpo acompaña platos intensos.')


# ── PROMPTS ──────────────────────────────────────────────────────────────
# @mcp_server.prompt() registra una plantilla reutilizable.
# Los prompts NO ejecutan acciones: devuelven texto configurado.
# El cliente los obtiene con get_prompt() y los usa con su LLM.

@mcp_server.prompt()
def prompt_chef_experto(tipo_cocina: str) -> str:
    'Plantilla que configura al asistente como chef experto en un tipo de cocina.'
    return (
        f'Eres un chef profesional con 20 años de experiencia en cocina {tipo_cocina}. '
        f'Tu misión es guiar al usuario para preparar platos auténticos de esta tradición. '
        f'Explica cada técnica con precisión y menciona los errores más comunes a evitar.'
    )


@mcp_server.prompt()
def prompt_adaptacion_dieta(tipo_dieta: str, restricciones: str) -> str:
    'Plantilla para adaptar recetas a un tipo de dieta y restricciones alimentarias.'
    return (
        f'Eres nutricionista y chef especializado en cocina {tipo_dieta}. '
        f'Restricciones del usuario: {restricciones}. '
        f'Sugiere sustitutos específicos manteniendo sabor y textura originales.'
    )


# ── ARRANCAR EL SERVIDOR ─────────────────────────────────────────────────
if __name__ == '__main__':
    mcp_server.run(transport='sse', host='0.0.0.0', port=8000)


## 🚀 PARTE 4 — Arrancar el servidor y verificar Tools y Prompts

Arrancamos el servidor como proceso separado, luego nos conectamos como cliente
MCP para verificar que las tools y prompts están correctamente registrados.


In [ ]:
# ============================================================
# CELDA 4.1 — Arrancar el servidor MCP v1
# ============================================================

import subprocess, socket, time

# Popen lanza el proceso sin bloquear el notebook.
# stdout/stderr en PIPE para capturar errores si el servidor falla.
servidor_proceso = subprocess.Popen(
    ['python', 'mcp_server.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print(f'✅ Proceso del servidor MCP iniciado (PID: {servidor_proceso.pid})')
print('   Esperando que uvicorn arranque en http://localhost:8000...')

# Verificamos que el puerto 8000 esté escuchando (máximo 30 segundos)
for intento in range(30):
    try:
        conn = socket.create_connection(('localhost', 8000), timeout=1)
        conn.close()  # Solo verificamos disponibilidad
        print(f'✅ Servidor MCP listo en http://localhost:8000 (tardó {intento + 1}s)')
        break
    except (socket.timeout, ConnectionRefusedError):
        time.sleep(1)
        if (intento + 1) % 10 == 0:
            print(f'   Esperando... {intento + 1}s')
else:
    err = servidor_proceso.stderr.read(500).decode('utf-8', errors='replace')
    print(f'❌ El servidor no arrancó en 30s. Errores:\n{err}')


In [ ]:
# ============================================================
# CELDA 4.2 — Listar las tools del servidor con list_tools()
# ============================================================
# list_tools() devuelve el catálogo con sus JSON Schemas.
# Así cualquier cliente MCP descubre qué puede hacer un servidor.

async def listar_herramientas():
    async with sse_client('http://localhost:8000/sse') as (leer, escribir):
        async with ClientSession(leer, escribir) as sesion:
            await sesion.initialize()              # Handshake: intercambia capacidades
            respuesta = await sesion.list_tools()  # Solicita el catálogo de tools
            return respuesta.tools

herramientas = asyncio.run(listar_herramientas())

print(f'✅ El servidor tiene {len(herramientas)} tools registradas:\n')
for tool in herramientas:
    print(f'  🔧 {tool.name}')
    print(f'     {tool.description}')
    if tool.inputSchema and 'properties' in tool.inputSchema:
        params = list(tool.inputSchema['properties'].keys())
        print(f'     Parámetros: {params}')  # JSON Schema generado por FastMCP
    print()


In [ ]:
# ============================================================
# CELDA 4.3 — Función auxiliar: llamar una tool del servidor
# ============================================================

async def llamar_tool(nombre_tool: str, argumentos: dict) -> str:
    'Conecta al servidor MCP, ejecuta la tool y devuelve el texto del resultado.'
    async with sse_client('http://localhost:8000/sse') as (leer, escribir):
        async with ClientSession(leer, escribir) as sesion:
            await sesion.initialize()
            resultado = await sesion.call_tool(nombre_tool, argumentos)
            return resultado.content[0].text if resultado.content else '(sin resultado)'

# Test 1: buscar_receta
print('=== TEST: buscar_receta(nombre=paella, num_personas=6) ===\n')
resp = asyncio.run(llamar_tool('buscar_receta', {'nombre': 'paella', 'num_personas': 6}))
print(resp[:600])
print('\n✅ Tool buscar_receta funciona correctamente')


In [ ]:
# ============================================================
# CELDA 4.4 — Test de calcular_calorias y sugerir_maridaje
# ============================================================

print('=== TEST: calcular_calorias(plato=carbonara, gramaje=350) ===\n')
resp = asyncio.run(llamar_tool('calcular_calorias', {'plato': 'carbonara', 'gramaje': 350}))
print(resp)

print()

print('=== TEST: sugerir_maridaje(plato=gazpacho) ===\n')
resp = asyncio.run(llamar_tool('sugerir_maridaje', {'plato': 'gazpacho'}))
print(resp)

print('\n✅ Las 3 tools básicas funcionan correctamente')


In [ ]:
# ============================================================
# CELDA 4.5 — Listar y obtener un Prompt del servidor
# ============================================================
# list_prompts() → catálogo de prompts disponibles
# get_prompt()   → mensajes renderizados con los argumentos dados

async def verificar_prompts():
    async with sse_client('http://localhost:8000/sse') as (leer, escribir):
        async with ClientSession(leer, escribir) as sesion:
            await sesion.initialize()

            # Listar prompts disponibles
            lista = await sesion.list_prompts()
            print(f'📋 El servidor tiene {len(lista.prompts)} prompts registrados:\n')
            for p in lista.prompts:
                print(f'  📝 {p.name}: {p.description}')
                if p.arguments:
                    print(f'     Argumentos: {[a.name for a in p.arguments]}')
                print()

            # Obtener el prompt con argumentos concretos
            print('=== Obteniendo prompt_chef_experto(tipo_cocina=italiana) ===\n')
            resultado = await sesion.get_prompt(
                'prompt_chef_experto',
                {'tipo_cocina': 'italiana'}  # Argumento del prompt
            )
            # El servidor devuelve mensajes listos para usar con cualquier LLM
            for msg in resultado.messages:
                print(f'  [{msg.role}]: {msg.content.text[:250]}...')

asyncio.run(verificar_prompts())
print('\n✅ Prompts verificados: list_prompts y get_prompt funcionan correctamente')


## 🧠 PARTE 5 — Sampling: el servidor delega texto a Ollama

**Sampling** es el mecanismo por el cual el servidor MCP le pide al cliente que
use su LLM para generar texto. El servidor no tiene LLM propio.

### ¿Por qué Sampling y no llamar Ollama desde el servidor?

```
Sin Sampling:                    Con Sampling:
  mcp_server.py                    mcp_server.py
  └── import ollama                └── ctx.session.create_message(...)
  └── acoplado al modelo               │
                                   Cliente (sampling_callback)
                                   └── llama a Ollama, Claude, GPT...
                                   └── el modelo se cambia SIN tocar el servidor
```

Con Sampling, el servidor es **agnóstico al LLM**: hoy usa Ollama,
mañana podría usar Claude API simplemente cambiando el `sampling_callback`.

### Flujo de Sampling

```
  call_tool('analizar_receta_con_ia', {'nombre': 'paella'})
       │
  Servidor: busca la receta, construye el prompt de análisis
  Servidor → ctx.session.create_message(messages, max_tokens=400)
  ─────────────────────────────────────────────────────────►
  Cliente recibe SamplingRequest
  sampling_callback → POST http://localhost:11434/api/chat
  Ollama genera análisis cultural y nutricional
  ◄─────────────────────────────────────────────────────────
  Servidor recibe el texto generado por Ollama
  Servidor construye la respuesta final
  ─────────────────────────────────────────────────────────►
  Cliente recibe el resultado de call_tool
```


In [ ]:
%%writefile mcp_server.py
# ============================================================
# mcp_server.py — Servidor MCP de Recetas v2 (añade Sampling)
# ============================================================
from mcp.server.fastmcp import FastMCP, Context
import mcp.types as types
from pydantic import BaseModel

mcp_server = FastMCP('chef-recetas')

RECETAS = {
    'paella': {'nombre': 'Paella Valenciana', 'origen': 'Valencia, España',
        'ingredientes': ['arroz bomba 400g', 'pollo 500g', 'conejo 300g', 'judías verdes 150g',
                         'azafrán 1 sobre', 'aceite de oliva 100ml'],
        'pasos': ['Calentar aceite', 'Dorar pollo y conejo 10 min',
                  'Añadir judías y tomate', 'Agregar arroz y caldo', 'Cocer 18 min'],
        'tiempo_min': 45, 'calorias_por_100g': 150, 'dificultad': 'media'},
    'tortilla': {'nombre': 'Tortilla Española', 'origen': 'España',
        'ingredientes': ['huevos 6 uds', 'patatas 500g', 'cebolla 1 ud', 'aceite 200ml'],
        'pasos': ['Cortar patatas fino', 'Freír en aceite bajo 20 min',
                  'Batir huevos y mezclar', 'Cuajar 4 min por lado'],
        'tiempo_min': 35, 'calorias_por_100g': 170, 'dificultad': 'fácil'},
    'gazpacho': {'nombre': 'Gazpacho Andaluz', 'origen': 'Andalucía, España',
        'ingredientes': ['tomates maduros 1kg', 'pepino 1 ud', 'pimiento verde 1 ud',
                         'ajo 2 dientes', 'pan 100g', 'aceite 100ml', 'vinagre 30ml'],
        'pasos': ['Remojar pan', 'Triturar todo', 'Colar pieles', 'Refrigerar 2h'],
        'tiempo_min': 20, 'calorias_por_100g': 35, 'dificultad': 'fácil'},
    'carbonara': {'nombre': 'Pasta Carbonara', 'origen': 'Roma, Italia',
        'ingredientes': ['spaghetti 400g', 'guanciale 200g', 'yemas 4 uds', 'pecorino 100g'],
        'pasos': ['Cocer pasta al dente', 'Dorar guanciale', 'Mezclar yemas y queso',
                  'Unir fuera del fuego', 'Añadir agua de pasta'],
        'tiempo_min': 25, 'calorias_por_100g': 200, 'dificultad': 'media'},
    'hummus': {'nombre': 'Hummus Clásico', 'origen': 'Oriente Medio',
        'ingredientes': ['garbanzos 400g', 'tahini 60ml', 'limón 3 cdas', 'ajo 2 dientes'],
        'pasos': ['Reservar aquafaba', 'Triturar ajo con sal',
                  'Añadir garbanzos y tahini', 'Agregar aquafaba hasta cremoso'],
        'tiempo_min': 15, 'calorias_por_100g': 165, 'dificultad': 'fácil'},
}


# ── TOOLS BÁSICAS ────────────────────────────────────────────────────────

@mcp_server.tool()
def buscar_receta(nombre: str, num_personas: int = 4) -> str:
    'Busca una receta de cocina y devuelve ingredientes y pasos.'
    receta = RECETAS.get(nombre.lower())
    if not receta:
        return f"No encontré '{nombre}'. Disponibles: {', '.join(RECETAS.keys())}"
    r = f"# {receta['nombre']}\n**Origen:** {receta['origen']} | **Tiempo:** {receta['tiempo_min']} min\n\n"
    r += f"## Ingredientes para {num_personas} personas\n"
    for ing in receta['ingredientes']:
        r += f'- {ing}\n'
    r += '\n## Preparación\n'
    for i, paso in enumerate(receta['pasos'], 1):
        r += f'{i}. {paso}\n'
    return r

@mcp_server.tool()
def calcular_calorias(plato: str, gramaje: int = 300) -> str:
    'Calcula las calorías aproximadas de una porción.'
    receta = RECETAS.get(plato.lower())
    if not receta:
        return f"Sin datos para '{plato}'."
    cal = int(receta['calorias_por_100g'] * gramaje / 100)
    return f"**{receta['nombre']}** — {gramaje}g: ~{cal} kcal ({'ligero ✅' if cal < 400 else 'contundente 🍽️'})"

@mcp_server.tool()
def sugerir_maridaje(plato: str) -> str:
    'Sugiere la bebida ideal para acompañar el plato.'
    maridajes = {'paella': 'Vino blanco valenciano', 'tortilla': 'Rioja joven',
                 'gazpacho': 'Fino de Jerez', 'carbonara': 'Frascati', 'hummus': 'Té de menta'}
    m = maridajes.get(plato.lower())
    return f'**Maridaje para {plato}:** {m}' if m else f"Sin maridaje para '{plato}'."


# ── PROMPTS ──────────────────────────────────────────────────────────────

@mcp_server.prompt()
def prompt_chef_experto(tipo_cocina: str) -> str:
    'Plantilla de chef experto para un tipo de cocina específico.'
    return (f'Eres un chef profesional con 20 años de experiencia en cocina {tipo_cocina}. '
            f'Guía al usuario para preparar platos auténticos con técnica y precisión.')

@mcp_server.prompt()
def prompt_adaptacion_dieta(tipo_dieta: str, restricciones: str) -> str:
    'Plantilla para adaptar recetas a dietas y restricciones alimentarias.'
    return (f'Eres nutricionista y chef de cocina {tipo_dieta}. '
            f'Restricciones: {restricciones}. Sugiere sustitutos manteniendo sabor y textura.')


# ── TOOL CON SAMPLING ────────────────────────────────────────────────────
# Esta tool demuestra Sampling: el servidor construye un prompt y pide al
# CLIENTE que use su LLM (Ollama) para generar el texto de análisis.

@mcp_server.tool()
async def analizar_receta_con_ia(nombre: str, ctx: Context) -> str:
    'Analiza en profundidad una receta usando IA mediante Sampling MCP.'
    receta = RECETAS.get(nombre.lower())
    if not receta:
        return f"No encontré '{nombre}'."

    # Construimos el prompt de análisis con los datos de la receta
    prompt_analisis = (
        f'Analiza esta receta brevemente:\n'
        f"Nombre: {receta['nombre']}, Origen: {receta['origen']}, "
        f"Ingredientes: {', '.join(receta['ingredientes'][:3])}, "
        f"Tiempo: {receta['tiempo_min']} min.\n\n"
        f'En 3 párrafos cortos: 1) contexto cultural, 2) técnica principal, 3) consejo clave.'
    )

    # SAMPLING: ctx.session.create_message() envía el prompt al cliente.
    # El cliente lo resuelve con su LLM (Ollama) y devuelve el texto generado.
    resultado = await ctx.session.create_message(
        messages=[
            types.SamplingMessage(
                role='user',  # El prompt va como mensaje de usuario
                content=types.TextContent(type='text', text=prompt_analisis)
            )
        ],
        max_tokens=400  # Limitamos la longitud para respuestas más rápidas
    )

    # resultado.content.text contiene el texto que Ollama generó en el cliente
    return (f'🤖 **Análisis de {receta["nombre"]}** (generado por Ollama via Sampling):\n\n'
            f'{resultado.content.text}')


if __name__ == '__main__':
    mcp_server.run(transport='sse', host='0.0.0.0', port=8000)


In [ ]:
# ============================================================
# CELDA 5.2 — Reiniciar el servidor con la versión de Sampling
# ============================================================

servidor_proceso.terminate()      # Enviamos señal SIGTERM al proceso
servidor_proceso.wait(timeout=5)  # Esperamos a que termine limpiamente
print(f'✅ Servidor anterior (PID {servidor_proceso.pid}) detenido')
time.sleep(1)  # Pausa breve para liberar el puerto 8000

servidor_proceso = subprocess.Popen(
    ['python', 'mcp_server.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

for intento in range(30):
    try:
        conn = socket.create_connection(('localhost', 8000), timeout=1)
        conn.close()
        print(f'✅ Servidor MCP v2 (con Sampling) listo (tardó {intento + 1}s)')
        break
    except (socket.timeout, ConnectionRefusedError):
        time.sleep(1)
else:
    err = servidor_proceso.stderr.read(500).decode('utf-8', errors='replace')
    print(f'❌ El servidor no arrancó.\nErrores:\n{err}')


In [ ]:
# ============================================================
# CELDA 5.3 — Definir el sampling_callback del cliente
# ============================================================
# El sampling_callback se ejecuta cuando el servidor pide una completación LLM.
# Recibe los mensajes del servidor y llama a Ollama.
# El servidor no sabe qué LLM se usa; ese es el punto de desacoplamiento.

async def sampling_callback(params) -> types.CreateMessageResult:
    'Recibe SamplingRequest del servidor MCP y lo resuelve con Ollama.'
    # params.messages: lista de mensajes que el servidor quiere procesar
    # params.maxTokens: límite de tokens que el servidor solicita al cliente

    # Convertimos los mensajes MCP al formato que espera la API HTTP de Ollama
    mensajes_ollama = []
    for msg in params.messages:
        if isinstance(msg.content, types.TextContent):  # Solo mensajes de texto
            mensajes_ollama.append({
                'role': msg.role,         # 'user' o 'assistant'
                'content': msg.content.text  # Texto del mensaje
            })

    # Llamamos a Ollama directamente por su API HTTP (no requiere librería)
    respuesta = requests.post(
        'http://localhost:11434/api/chat',
        json={
            'model': MODELO_OLLAMA,              # El modelo configurado en la Parte 1
            'messages': mensajes_ollama,          # Los mensajes del servidor
            'stream': False,                      # Esperamos respuesta completa
            'options': {'num_predict': params.maxTokens or 300}  # Respetamos el límite
        },
        timeout=120  # Ollama puede tardar hasta 2 minutos en T4
    )
    respuesta.raise_for_status()  # Lanza excepción si Ollama devolvió error HTTP
    texto_generado = respuesta.json()['message']['content']  # Texto generado

    # Construimos el CreateMessageResult que el servidor espera
    return types.CreateMessageResult(
        role='assistant',                                      # Respuesta del asistente
        content=types.TextContent(type='text', text=texto_generado),
        model=MODELO_OLLAMA,                                   # Informamos qué modelo usamos
        stopReason='endTurn'                                   # Razón de parada estándar
    )

print('✅ sampling_callback definido')
print(f'   Cuando el servidor pida Sampling → cliente llama Ollama ({MODELO_OLLAMA})')


In [ ]:
# ============================================================
# CELDA 5.4 — Test de Sampling: analizar_receta_con_ia
# ============================================================
# Al llamar esta tool:
# 1. Servidor busca la receta y construye el prompt de análisis
# 2. Servidor envía SamplingRequest al cliente
# 3. sampling_callback llama a Ollama con el prompt
# 4. Ollama genera el análisis (puede tardar 15-30 segundos)
# 5. Cliente devuelve el texto al servidor
# 6. Servidor construye la respuesta final de la tool

async def llamar_tool_con_sampling(nombre_tool: str, argumentos: dict) -> str:
    'Llama una tool con sampling_callback activo.'
    async with sse_client('http://localhost:8000/sse') as (leer, escribir):
        async with ClientSession(
            leer, escribir,
            sampling_callback=sampling_callback  # Registramos el callback de Sampling
        ) as sesion:
            # Durante initialize(), el cliente anuncia al servidor que soporta Sampling.
            # Sin esto, ctx.session.create_message() fallaría en el servidor.
            await sesion.initialize()
            resultado = await sesion.call_tool(nombre_tool, argumentos)
            return resultado.content[0].text if resultado.content else '(sin resultado)'

print('=== TEST SAMPLING: analizar_receta_con_ia(nombre=tortilla) ===')
print('(Ollama tardará 15-30s en generar el análisis...)\n')

analisis = asyncio.run(llamar_tool_con_sampling(
    'analizar_receta_con_ia',
    {'nombre': 'tortilla'}
))
print(analisis)
print('\n✅ Sampling verificado: el servidor delegó la generación de texto a Ollama')


## 📝 PARTE 6 — Elicitation: el servidor solicita datos al usuario

**Elicitation** es el inverso conceptual de Sampling:
- **Sampling**: el servidor necesita texto generado por IA → pide al cliente su LLM
- **Elicitation**: el servidor necesita datos del usuario → pide al cliente que muestre un formulario

### Restricción del schema de Elicitation

El schema solo acepta tipos primitivos: `str`, `int`, `float`, `bool`, `list[str]`.
Esta restricción garantiza que cualquier cliente (Gradio, CLI, app móvil) pueda mostrar el formulario.

### Flujo de Elicitation

```
  call_tool('adaptar_receta', {'nombre': 'paella'})
       │
  Servidor verifica la receta; necesita datos de personalización
  Servidor → ctx.elicit(mensaje, schema=DatosAdaptacion)
  ──────────────────────────────────────────────────────────────►
  Cliente recibe ElicitRequest con el schema
  elicitation_callback → muestra formulario → usuario rellena
  ◄──────────────────────────────────────────────────────────────
  Servidor recibe ElicitResult.data = {num_personas, alergias, nivel_picante}
  Servidor adapta la receta con esos datos
  ──────────────────────────────────────────────────────────────►
  Cliente recibe la receta personalizada
```


In [ ]:
%%writefile mcp_server.py
# ============================================================
# mcp_server.py — Servidor MCP de Recetas FINAL
# ============================================================
# Versión completa: 3 tools básicas + Sampling + Elicitation + 2 Prompts
from mcp.server.fastmcp import FastMCP, Context
import mcp.types as types
from pydantic import BaseModel

mcp_server = FastMCP('chef-recetas')

RECETAS = {
    'paella': {'nombre': 'Paella Valenciana', 'origen': 'Valencia, España',
        'ingredientes': ['arroz bomba 400g', 'pollo 500g', 'conejo 300g', 'judías verdes 150g',
                         'azafrán 1 sobre', 'aceite de oliva 100ml', 'sal al gusto'],
        'pasos': ['Calentar aceite en paellera', 'Dorar pollo y conejo 10 min',
                  'Añadir judías verdes 5 min', 'Incorporar tomate rallado 5 min',
                  'Agregar arroz y caldo (doble)', 'Cocer 18 min sin remover'],
        'tiempo_min': 45, 'calorias_por_100g': 150, 'dificultad': 'media',
        'maridaje': 'Vino blanco valenciano DO Valencia'},
    'tortilla': {'nombre': 'Tortilla Española', 'origen': 'España',
        'ingredientes': ['huevos 6 uds', 'patatas 500g', 'cebolla 1 ud', 'aceite 200ml', 'sal'],
        'pasos': ['Cortar patatas fino', 'Freír en aceite bajo 20 min',
                  'Batir huevos y mezclar', 'Cuajar 4 min por lado'],
        'tiempo_min': 35, 'calorias_por_100g': 170, 'dificultad': 'fácil',
        'maridaje': 'Vino tinto Rioja joven'},
    'gazpacho': {'nombre': 'Gazpacho Andaluz', 'origen': 'Andalucía, España',
        'ingredientes': ['tomates maduros 1kg', 'pepino 1 ud', 'pimiento verde 1 ud',
                         'ajo 2 dientes', 'pan 100g', 'aceite 100ml', 'vinagre 30ml'],
        'pasos': ['Remojar pan en agua fría', 'Triturar todo hasta crema',
                  'Colar pieles y pepitas', 'Refrigerar mínimo 2 horas'],
        'tiempo_min': 20, 'calorias_por_100g': 35, 'dificultad': 'fácil',
        'maridaje': 'Fino de Jerez muy frío'},
    'carbonara': {'nombre': 'Pasta Carbonara', 'origen': 'Roma, Italia',
        'ingredientes': ['spaghetti 400g', 'guanciale 200g', 'yemas de huevo 4 uds',
                         'pecorino romano 100g', 'pimienta negra', 'sal'],
        'pasos': ['Cocer pasta al dente, reservar agua', 'Dorar guanciale sin aceite',
                  'Mezclar yemas con queso y pimienta', 'Unir pasta + guanciale fuera del fuego',
                  'Añadir mezcla de yemas con agua de pasta'],
        'tiempo_min': 25, 'calorias_por_100g': 200, 'dificultad': 'media',
        'maridaje': 'Vino blanco italiano Frascati'},
    'hummus': {'nombre': 'Hummus Clásico', 'origen': 'Oriente Medio',
        'ingredientes': ['garbanzos cocidos 400g', 'tahini 60ml', 'limón (zumo) 3 cdas',
                         'ajo 2 dientes', 'aceite de oliva 3 cdas', 'comino 1 cdta', 'sal'],
        'pasos': ['Reservar aquafaba', 'Triturar ajo con sal',
                  'Añadir garbanzos, tahini, limón y comino; triturar',
                  'Agregar aquafaba hasta textura cremosa'],
        'tiempo_min': 15, 'calorias_por_100g': 165, 'dificultad': 'fácil',
        'maridaje': 'Té de menta caliente'},
}


# ── TOOLS BÁSICAS ────────────────────────────────────────────────────────

@mcp_server.tool()
def buscar_receta(nombre: str, num_personas: int = 4) -> str:
    'Busca una receta de cocina y devuelve ingredientes y pasos.'
    receta = RECETAS.get(nombre.lower())
    if not receta:
        return f"No encontré '{nombre}'. Disponibles: {', '.join(RECETAS.keys())}"
    r = f"# {receta['nombre']}\n**Origen:** {receta['origen']} | **Tiempo:** {receta['tiempo_min']} min\n\n## Ingredientes para {num_personas} personas\n"
    for ing in receta['ingredientes']:
        r += f'- {ing}\n'
    r += '\n## Preparación\n'
    for i, paso in enumerate(receta['pasos'], 1):
        r += f'{i}. {paso}\n'
    return r

@mcp_server.tool()
def calcular_calorias(plato: str, gramaje: int = 300) -> str:
    'Calcula las calorías aproximadas de una porción.'
    receta = RECETAS.get(plato.lower())
    if not receta:
        return f"Sin datos para '{plato}'."
    cal = int(receta['calorias_por_100g'] * gramaje / 100)
    return f"**{receta['nombre']}** — {gramaje}g: ~{cal} kcal ({'ligero ✅' if cal < 400 else 'contundente 🍽️'})"

@mcp_server.tool()
def sugerir_maridaje(plato: str) -> str:
    'Sugiere la bebida ideal para acompañar el plato.'
    receta = RECETAS.get(plato.lower())
    if not receta:
        return f"Sin maridaje para '{plato}'."
    return f"**Maridaje para {receta['nombre']}:** {receta['maridaje']}"


# ── PROMPTS ──────────────────────────────────────────────────────────────

@mcp_server.prompt()
def prompt_chef_experto(tipo_cocina: str) -> str:
    'Plantilla de chef experto parametrizada por tipo de cocina.'
    return (f'Eres un chef profesional con 20 años de experiencia en cocina {tipo_cocina}. '
            f'Guía al usuario para preparar platos auténticos con técnica y precisión.')

@mcp_server.prompt()
def prompt_adaptacion_dieta(tipo_dieta: str, restricciones: str) -> str:
    'Plantilla para adaptar recetas a dietas y restricciones alimentarias.'
    return (f'Eres nutricionista y chef de cocina {tipo_dieta}. '
            f'Restricciones: {restricciones}. Sugiere sustitutos manteniendo sabor y textura.')


# ── TOOL CON SAMPLING ────────────────────────────────────────────────────

@mcp_server.tool()
async def analizar_receta_con_ia(nombre: str, ctx: Context) -> str:
    'Analiza en profundidad una receta usando IA mediante Sampling MCP.'
    receta = RECETAS.get(nombre.lower())
    if not receta:
        return f"No encontré '{nombre}'."
    prompt_analisis = (
        f"Analiza esta receta brevemente:\nNombre: {receta['nombre']}, "
        f"Origen: {receta['origen']}, Ingredientes: {', '.join(receta['ingredientes'][:3])}, "
        f"Tiempo: {receta['tiempo_min']} min.\n\n"
        f'En 3 párrafos: 1) contexto cultural, 2) técnica principal, 3) consejo clave.'
    )
    # SAMPLING: pedimos al cliente que use su LLM para generar el análisis.
    # El servidor no importa Ollama; el cliente decide qué modelo usar.
    resultado = await ctx.session.create_message(
        messages=[types.SamplingMessage(
            role='user',
            content=types.TextContent(type='text', text=prompt_analisis)
        )],
        max_tokens=400
    )
    return (f'🤖 **Análisis de {receta["nombre"]}** (Ollama via Sampling):\n\n'
            f'{resultado.content.text}')


# ── TOOL CON ELICITATION ─────────────────────────────────────────────────

class DatosAdaptacion(BaseModel):
    'Schema de datos que el servidor solicita al usuario para adaptar la receta.'
    # Solo tipos primitivos: str, int, float, bool, list[str]
    # Restricción del protocolo MCP para garantizar compatibilidad con cualquier cliente
    num_personas: int   # Número de personas (ajusta cantidades)
    alergias: str       # Ingredientes a evitar ('ninguna' si no hay restricciones)
    nivel_picante: str  # 'sin picante', 'suave' o 'picante'


@mcp_server.tool()
async def adaptar_receta(nombre: str, ctx: Context) -> str:
    'Adapta una receta a preferencias personales mediante Elicitation MCP.'
    receta = RECETAS.get(nombre.lower())
    if not receta:
        return f"No encontré '{nombre}'. Disponibles: {list(RECETAS.keys())}"

    # ELICITATION: pedimos al cliente los datos de personalización.
    # El servidor define el schema (qué campos necesita).
    # El cliente define cómo obtenerlos (formulario Gradio, CLI, app móvil...).
    resultado_elicit = await ctx.elicit(
        message=f"Para personalizar '{receta['nombre']}' necesito algunos datos:",
        schema=DatosAdaptacion  # Schema Pydantic con los campos del formulario
    )

    # El resultado puede ser: accept (usuario aceptó), decline o cancel
    if resultado_elicit.action == 'decline':
        return 'El usuario declinó proporcionar los datos.'
    if resultado_elicit.action == 'cancel':
        return 'Operación cancelada por el usuario.'

    # action == 'accept': el objeto data tiene los campos del schema
    datos = resultado_elicit.data  # Instancia de DatosAdaptacion
    num_personas = datos.num_personas
    alergias = datos.alergias
    nivel_picante = datos.nivel_picante

    # Construimos la receta personalizada con los datos recibidos
    r = f"# {receta['nombre']} — Personalizada para {num_personas} personas\n\n"
    if alergias.lower() not in ('ninguna', 'none', 'no'):
        r += f'⚠️ **Atención — evitar: {alergias}**\n'
        r += f'Revisa y sustituye ingredientes que contengan {alergias}.\n\n'
    if nivel_picante == 'sin picante':
        r += '🌶️ Sin picante: omite guindillas, cayena y pimentón picante.\n'
    elif nivel_picante == 'picante':
        r += '🌶️🌶️ Versión picante: añade 1 guindilla o ½ cdta de cayena.\n'
    else:
        r += '🌶️ Nivel de picante: suave (estándar de la receta).\n'
    r += f"\n## Ingredientes para {num_personas} personas:\n"
    for ing in receta['ingredientes']:
        r += f'- {ing}\n'
    r += f'\n✅ Receta adaptada correctamente (Elicitation action: {resultado_elicit.action})'
    return r


if __name__ == '__main__':
    mcp_server.run(transport='sse', host='0.0.0.0', port=8000)


In [ ]:
# ============================================================
# CELDA 6.2 — Reiniciar con el servidor final
# ============================================================

servidor_proceso.terminate()
servidor_proceso.wait(timeout=5)
print(f'✅ Servidor anterior detenido')
time.sleep(1)

servidor_proceso = subprocess.Popen(
    ['python', 'mcp_server.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

for intento in range(30):
    try:
        conn = socket.create_connection(('localhost', 8000), timeout=1)
        conn.close()
        print(f'✅ Servidor MCP final listo (tardó {intento + 1}s)')
        print('   Capacidades: 3 tools + Sampling + Elicitation + 2 Prompts')
        break
    except (socket.timeout, ConnectionRefusedError):
        time.sleep(1)
else:
    err = servidor_proceso.stderr.read(500).decode('utf-8', errors='replace')
    print(f'❌ Error al arrancar.\n{err}')


In [ ]:
# ============================================================
# CELDA 6.3 — Definir el elicitation_callback del cliente
# ============================================================
# El elicitation_callback se ejecuta cuando el servidor pide datos al usuario.
# Recibe el mensaje y el schema del servidor, devuelve ElicitResult con los datos.

# Variable compartida: Gradio actualiza estos valores antes de llamar la tool.
# El callback los lee cuando el servidor llama ctx.elicit().
datos_adaptacion_pendientes = {
    'num_personas': 4,
    'alergias': 'ninguna',
    'nivel_picante': 'suave'
}

async def elicitation_callback(params) -> types.ElicitResult:
    'Recibe ElicitRequest del servidor y devuelve los datos del usuario.'
    # params.message: el texto descriptivo del servidor para el usuario
    # params.requestedSchema: JSON Schema de los campos que el servidor necesita
    print(f'   [elicitation_callback] El servidor pide: {params.message}')
    campos = list(params.requestedSchema.get('properties', {}).keys())
    print(f'   [elicitation_callback] Campos solicitados: {campos}')

    # Devolvemos los datos que Gradio habrá configurado en datos_adaptacion_pendientes
    return types.ElicitResult(
        action='accept',                       # El usuario acepta proporcionar los datos
        content=datos_adaptacion_pendientes    # Los valores del formulario Gradio
    )

print('✅ elicitation_callback definido')
print(f'   Datos actuales: {datos_adaptacion_pendientes}')


In [ ]:
# ============================================================
# CELDA 6.4 — Test de Elicitation: adaptar_receta
# ============================================================
# Simulamos que el usuario rellenó el formulario con estos valores.
# En la Parte 7, Gradio actualiza datos_adaptacion_pendientes desde el UI.

datos_adaptacion_pendientes['num_personas'] = 8
datos_adaptacion_pendientes['alergias'] = 'gluten'
datos_adaptacion_pendientes['nivel_picante'] = 'sin picante'

async def llamar_tool_con_elicitation(nombre_tool: str, argumentos: dict) -> str:
    'Llama una tool con ambos callbacks (sampling y elicitation) activos.'
    async with sse_client('http://localhost:8000/sse') as (leer, escribir):
        async with ClientSession(
            leer, escribir,
            sampling_callback=sampling_callback,       # Para analizar_receta_con_ia
            elicitation_callback=elicitation_callback  # Para adaptar_receta
        ) as sesion:
            await sesion.initialize()
            resultado = await sesion.call_tool(nombre_tool, argumentos)
            return resultado.content[0].text if resultado.content else '(sin resultado)'

print('=== TEST ELICITATION: adaptar_receta(nombre=gazpacho) ===\n')
print('El servidor llamará al elicitation_callback con el schema de DatosAdaptacion.')
print('El callback responderá con: num_personas=8, alergias=gluten, nivel_picante=sin picante\n')

resultado = asyncio.run(llamar_tool_con_elicitation('adaptar_receta', {'nombre': 'gazpacho'}))
print(resultado)
print('\n✅ Elicitation verificado: el servidor solicitó datos y los recibió via callback')


## 🎨 PARTE 7 — Interfaz Gradio completa con 4 pestañas

| Pestaña | Concepto MCP | Qué hace el usuario |
|---------|-------------|---------------------|
| 🔧 Tools básicas | Tools | Busca recetas, calcula calorías, pide maridaje |
| 🧠 Análisis con IA | Sampling | Pide un análisis generado por Ollama |
| ✏️ Adaptar receta | Elicitation | Rellena num_personas, alergias y picante |
| 📋 Prompts | Prompts | Obtiene plantillas parametrizadas del servidor |

### ¿Cómo funciona la Elicitation en Gradio?

El usuario rellena el formulario **antes** de pulsar 'Adaptar'. Cuando pulsa el botón,
actualizamos `datos_adaptacion_pendientes` y llamamos la tool. El servidor llama al
`elicitation_callback`, que ya tiene los datos listos y los devuelve inmediatamente.


In [ ]:
# ============================================================
# CELDA 7.1 — Interfaz Gradio con 4 pestañas MCP
# ============================================================

import gradio as gr

RECETAS_OPCIONES = ['paella', 'tortilla', 'gazpacho', 'carbonara', 'hummus']

# ── Callbacks de las pestañas ─────────────────────────────────────────

def fn_buscar_receta(nombre, num_personas):
    return asyncio.run(llamar_tool('buscar_receta', {'nombre': nombre, 'num_personas': int(num_personas)}))

def fn_calcular_calorias(plato, gramaje):
    return asyncio.run(llamar_tool('calcular_calorias', {'plato': plato, 'gramaje': int(gramaje)}))

def fn_sugerir_maridaje(plato):
    return asyncio.run(llamar_tool('sugerir_maridaje', {'plato': plato}))

def fn_analizar_receta(nombre):
    'Llama la tool de Sampling; Ollama puede tardar 15-30 segundos.'
    return asyncio.run(llamar_tool_con_sampling('analizar_receta_con_ia', {'nombre': nombre}))

def fn_adaptar_receta(nombre, num_personas, alergias, nivel_picante):
    'Actualiza datos_adaptacion_pendientes y luego llama la tool de Elicitation.'
    datos_adaptacion_pendientes['num_personas'] = int(num_personas)
    datos_adaptacion_pendientes['alergias'] = alergias
    datos_adaptacion_pendientes['nivel_picante'] = nivel_picante
    return asyncio.run(llamar_tool_con_elicitation('adaptar_receta', {'nombre': nombre}))

def fn_obtener_prompt(nombre_prompt, arg1, arg2):
    'Obtiene un prompt renderizado del servidor MCP.'
    async def _get():
        async with sse_client('http://localhost:8000/sse') as (leer, escribir):
            async with ClientSession(leer, escribir) as sesion:
                await sesion.initialize()
                args = ({'tipo_cocina': arg1} if nombre_prompt == 'prompt_chef_experto'
                        else {'tipo_dieta': arg1, 'restricciones': arg2})
                resultado = await sesion.get_prompt(nombre_prompt, args)
                return '\n\n'.join(f'[{m.role}]: {m.content.text}' for m in resultado.messages)
    return asyncio.run(_get())

# ── Construcción de la interfaz ───────────────────────────────────────

with gr.Blocks(title='🍳 MCP Chef — 4 mecanismos del SDK de Anthropic',
               theme=gr.themes.Soft()) as demo:

    gr.Markdown('# 🍳 MCP Chef — Los 4 mecanismos del SDK de Anthropic (mcp package)')
    gr.Markdown('Cada pestaña demuestra un mecanismo diferente. El servidor corre en el puerto 8000.')

    with gr.Tabs():

        # ── PESTAÑA 1: Tools básicas ─────────────────────────────────
        with gr.Tab('🔧 Tools básicas'):
            gr.Markdown('### Mecanismo: **Tools MCP** — El cliente invoca, el servidor ejecuta con datos locales.')
            with gr.Row():
                with gr.Column():
                    gr.Markdown('#### Buscar receta')
                    t1_nombre = gr.Dropdown(RECETAS_OPCIONES, label='Receta', value='paella')
                    t1_personas = gr.Slider(1, 12, value=4, step=1, label='Número de personas')
                    t1_btn = gr.Button('Buscar receta', variant='primary')
                with gr.Column():
                    gr.Markdown('#### Calorías y maridaje')
                    t1_plato = gr.Dropdown(RECETAS_OPCIONES, label='Plato', value='paella')
                    t1_gramaje = gr.Slider(100, 600, value=300, step=50, label='Gramaje (g)')
                    t1_cal_btn = gr.Button('Calcular calorías', variant='secondary')
                    t1_mar_btn = gr.Button('Sugerir maridaje', variant='secondary')
            t1_out = gr.Markdown()
            t1_btn.click(fn_buscar_receta, [t1_nombre, t1_personas], t1_out)
            t1_cal_btn.click(fn_calcular_calorias, [t1_plato, t1_gramaje], t1_out)
            t1_mar_btn.click(fn_sugerir_maridaje, [t1_plato], t1_out)

        # ── PESTAÑA 2: Sampling ──────────────────────────────────────
        with gr.Tab('🧠 Análisis con IA (Sampling)'):
            gr.Markdown(
                '### Mecanismo: **Sampling MCP**\n'
                'El servidor le pide al cliente que use Ollama para generar texto. '
                'El servidor no tiene LLM propio; el cliente decide qué modelo usar.'
            )
            gr.Markdown(
                '```\n'
                'Gradio → call_tool(analizar_receta_con_ia) → Servidor\n'
                '                                                 │ ctx.session.create_message()\n'
                'Ollama ← sampling_callback ← Cliente ← SamplingRequest\n'
                '                                                 │\n'
                'Gradio ← resultado final          ← texto generado ←┘\n'
                '```'
            )
            t2_nombre = gr.Dropdown(RECETAS_OPCIONES, label='Receta a analizar', value='paella')
            t2_btn = gr.Button('Analizar con IA (tardará 15-30s)', variant='primary')
            t2_out = gr.Markdown()
            t2_btn.click(fn_analizar_receta, [t2_nombre], t2_out)

        # ── PESTAÑA 3: Elicitation ───────────────────────────────────
        with gr.Tab('✏️ Adaptar receta (Elicitation)'):
            gr.Markdown(
                '### Mecanismo: **Elicitation MCP**\n'
                'El servidor solicita datos estructurados al usuario a través del cliente. '
                'Define un schema con los campos que necesita; Gradio los captura.'
            )
            with gr.Row():
                with gr.Column():
                    t3_nombre = gr.Dropdown(RECETAS_OPCIONES, label='Receta a adaptar', value='paella')
                    t3_personas = gr.Slider(1, 20, value=4, step=1, label='Número de personas')
                    t3_alergias = gr.Textbox(
                        label='Alergias o ingredientes a evitar',
                        placeholder='p.ej. gluten, lactosa o ninguna',
                        value='ninguna'
                    )
                    t3_picante = gr.Radio(
                        ['sin picante', 'suave', 'picante'],
                        label='Nivel de picante', value='suave'
                    )
                    t3_btn = gr.Button('Adaptar receta', variant='primary')
                with gr.Column():
                    gr.Markdown(
                        '#### Schema de Elicitation (DatosAdaptacion):\n'
                        '```python\n'
                        'class DatosAdaptacion(BaseModel):\n'
                        '    num_personas: int\n'
                        '    alergias: str\n'
                        '    nivel_picante: str\n'
                        '```\n'
                        'El servidor define este schema con `ctx.elicit()`.\n'
                        'El cliente Gradio rellena los campos con el formulario.'
                    )
            t3_out = gr.Markdown()
            t3_btn.click(fn_adaptar_receta, [t3_nombre, t3_personas, t3_alergias, t3_picante], t3_out)

        # ── PESTAÑA 4: Prompts ───────────────────────────────────────
        with gr.Tab('📋 Prompts MCP'):
            gr.Markdown(
                '### Mecanismo: **Prompts MCP**\n'
                'El servidor expone plantillas parametrizadas. El cliente las obtiene '
                'con `get_prompt()` y las usa para configurar conversaciones con Ollama.'
            )
            with gr.Row():
                with gr.Column():
                    t4_prompt = gr.Radio(
                        ['prompt_chef_experto', 'prompt_adaptacion_dieta'],
                        label='Prompt a obtener', value='prompt_chef_experto'
                    )
                    t4_arg1 = gr.Textbox(
                        label='Arg 1 (tipo_cocina o tipo_dieta)',
                        value='italiana'
                    )
                    t4_arg2 = gr.Textbox(
                        label='Arg 2 (restricciones — solo para prompt_adaptacion_dieta)',
                        value='sin gluten ni lactosa'
                    )
                    t4_btn = gr.Button('Obtener prompt del servidor', variant='primary')
                with gr.Column():
                    gr.Markdown(
                        'El texto devuelto es un **system prompt listo para usar** con cualquier LLM.\n'
                        'En un agente real, lo pasarías como mensaje de sistema al inicio de la conversación.'
                    )
            t4_out = gr.Markdown()
            t4_btn.click(fn_obtener_prompt, [t4_prompt, t4_arg1, t4_arg2], t4_out)

demo.launch(server_name='0.0.0.0', server_port=7860, share=False)
print('✅ Interfaz Gradio lanzada en http://localhost:7860')


## 💬 PARTE 8 — Reflexión

Responde las siguientes preguntas editando esta celda.

---

### Pregunta 1 — Conceptual
**¿Cuál es la diferencia entre Sampling y Elicitation en MCP? Da un ejemplo concreto de cuándo usarías cada uno en una aplicación real.**

> _Escribe tu respuesta aquí_

---

### Pregunta 2 — Código
**En la Parte 5, el `sampling_callback` recibe `params` y devuelve un `CreateMessageResult`. ¿Qué información contiene `params.messages` y por qué el servidor no puede llamar directamente a Ollama en lugar de usar Sampling?**

> _Escribe tu respuesta aquí_

---

### Pregunta 3 — Arquitectura
**El schema de Elicitation solo usa tipos primitivos (`str`, `int`). ¿Por qué crees que el protocolo MCP impone esta restricción? ¿Qué problema evita?**

> _Escribe tu respuesta aquí_

---

### Pregunta 4 — Prompts vs Hardcoding
**¿Qué ventaja tiene definir prompts en el servidor MCP con `@mcp.prompt()` en lugar de hardcodear el texto del system prompt en el cliente? Piensa en un equipo con 10 productos usando el mismo servidor.**

> _Escribe tu respuesta aquí_

---

### Pregunta 5 — Implicaciones
**El patrón Sampling invierte el control: el servidor pide al cliente usar su LLM. ¿Qué ventajas de privacidad y seguridad tiene este diseño frente a un servidor que llama directamente a una API externa?**

> _Escribe tu respuesta aquí_


## 🚀 PARTE 9 — Retos opcionales

### Reto 1 — Prompt con múltiples mensajes

El prompt `prompt_chef_experto` devuelve un `str`. Modifícalo para devolver una lista de mensajes
con un mensaje de usuario y una respuesta inicial del chef (mensaje de asistente).

**Pista:** Importa `UserMessage`, `AssistantMessage` desde `mcp.server.fastmcp.prompts.base`
en el servidor. El tipo de retorno cambia a `list[Message]`. Reinicia el servidor y verifica
que `get_prompt()` devuelve dos mensajes en lugar de uno.


In [ ]:
# Reto 1 — Tu código aquí

### Reto 2 — Elicitation con acción decline

Añade un botón 'Cancelar adaptación' en Gradio. Al pulsarlo antes de confirmar,
el `elicitation_callback` devuelve `ElicitResult(action='decline')`.

**Pista:** Añade una variable global `elicitation_accion = 'accept'`. El botón 'Cancelar'
la cambia a `'decline'`. El callback lee esta variable para decidir qué devolver.
Restablece a `'accept'` tras cada llamada.


In [ ]:
# Reto 2 — Tu código aquí

### Reto 3 — Panel de observabilidad MCP

Añade una quinta pestaña 'Observabilidad' con un log en tiempo real de todas las operaciones:
cada `call_tool` (nombre, duración), cada `SamplingRequest` (tokens solicitados)
y cada `ElicitRequest` (campos pedidos, acción devuelta).

**Pista:** Crea `log_mcp = []` global. Añade registros en las funciones `fn_*` de Gradio.
Usa `gr.Dataframe` para mostrar el historial y un botón 'Refrescar log'.


In [ ]:
# Reto 3 — Tu código aquí

## 🌐 PARTE 10 — Publicar con Ngrok

Cerramos la interfaz local y la publicamos con URL pública.

> **Requisito:** Token de Ngrok en Colab Secrets con el nombre `NGROK_TOKEN`.


In [ ]:
# ============================================================
# CELDA 10.1 — Importar pyngrok
# ============================================================
from pyngrok import ngrok, conf
from google.colab import userdata
print('✅ pyngrok importado correctamente')


In [ ]:
# ============================================================
# CELDA 10.2 — Autenticar con el token de Ngrok
# ============================================================
try:
    ngrok_token = userdata.get('NGROK_TOKEN')      # Lee desde Colab Secrets (nunca hardcodeado)
    conf.get_default().auth_token = ngrok_token
    print('✅ Token de Ngrok configurado correctamente')
except Exception as e:
    print(f'❌ No se encontró NGROK_TOKEN en Colab Secrets: {e}')
    print('   Ve a 🔑 Secrets y añade NGROK_TOKEN con tu token de ngrok.com')


In [ ]:
# ============================================================
# CELDA 10.3 — Publicar con Ngrok
# ============================================================
demo.close()   # Libera el puerto 7860
ngrok.kill()   # Cierra túneles anteriores

tunel = ngrok.connect(7860)  # Crea túnel HTTP al exterior
print(f'✅ Interfaz pública: {tunel.public_url}')
print('   Comparte esta URL para que tus compañeros prueben las 4 pestañas MCP')

demo.launch(
    server_name='0.0.0.0',  # Acepta conexiones locales (Ngrok enruta el tráfico externo)
    server_port=7860,
    share=False             # share=True es inestable; Ngrok es la solución profesional
)


In [ ]:
# ============================================================
# CELDA 10.4 — Verificar túneles activos
# ============================================================
tuneles = ngrok.get_tunnels()
if tuneles:
    print(f'✅ {len(tuneles)} túnel(es) activo(s):')
    for t in tuneles:
        print(f"   {t.name}: {t.public_url} → {t.config['addr']}")
else:
    print('❌ No hay túneles activos. Vuelve a ejecutar la celda anterior.')


In [ ]:
# ============================================================
# CELDA LIMPIEZA — Liberar todos los recursos al terminar
# ============================================================

# 1. Cerrar el túnel Ngrok
ngrok.kill()
print('✅ Túnel Ngrok cerrado')

# 2. Cerrar la interfaz Gradio
demo.close()
print('✅ Interfaz Gradio cerrada')

# 3. Terminar el servidor MCP
servidor_proceso.terminate()
servidor_proceso.wait(timeout=5)  # Esperamos a que termine limpiamente
print(f'✅ Servidor MCP (PID {servidor_proceso.pid}) detenido')

# 4. Terminar el servidor Ollama
ollama_proc.terminate()
ollama_proc.wait(timeout=5)
print('✅ Servidor Ollama detenido')

print('\n✅ Todos los recursos liberados. Puedes cerrar el notebook.')


## 📊 Rúbrica de Evaluación

| Criterio | Excelente (5) | Satisfactorio (3) | En desarrollo (1) |
|----------|--------------|-------------------|-------------------|
| **Tools básicas** | Las 3 tools funcionan para todas las recetas | Al menos 2 tools funcionan | El servidor arranca pero las tools dan error |
| **Prompts** | `list_prompts` muestra 2 prompts y `get_prompt` devuelve texto renderizado | Un prompt funciona | Prompts definidos pero `get_prompt` falla |
| **Sampling** | `analizar_receta_con_ia` genera análisis reales via Ollama; flujo server→callback visible | La tool funciona pero sin evidencia del flujo | La tool falla o no usa el sampling_callback |
| **Elicitation** | `adaptar_receta` personaliza con datos del formulario Gradio | La tool adapta con datos hardcodeados | La tool falla o ignora el elicitation_callback |
| **Interfaz Gradio** | 4 pestañas funcionales, cada una demuestra un mecanismo diferente | 3 pestañas funcionales | Gradio lanza pero los botones no responden |
| **Publicación Ngrok** | URL pública compartida y accesible desde otro dispositivo | Ngrok conecta pero sin verificación | `ngrok.connect()` ejecutado sin URL compartida |
